In [ ]:
# =========================================
# 1. IMPORT LIBRARIES
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay, roc_curve, auc
)


In [ ]:
# =========================================
# 2. LOAD DATA
# =========================================
df = pd.read_csv(r"../data/Dataset_actual.csv")

In [ ]:
# =========================================
# 3. PREPROCESSING
# =========================================

# Encode categorical columns
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

# Split features and target
X = df.drop("theft", axis=1)   
y = df["theft"]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
df.columns

In [ ]:
# =========================================
# 4. DEFINE MODELS
# =========================================

knn = KNeighborsClassifier()
rf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300)

models = {
    "KNN": knn,
    "Random Forest": rf,
    "XGBoost": xgb,
    "MLP": mlp
}

results = []

In [ ]:
# =========================================
# 5. TRAIN & EVALUATE MODELS
# =========================================

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc_score = roc_auc_score(y_test, y_prob, multi_class='ovr')

    results.append([name, acc, prec, rec, f1, auc_score])

    print(f"{name} Results:")
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1 Score:", f1)
    print("AUC-ROC:", auc_score)

In [ ]:
# =========================================
# 6. VOTING ENSEMBLE
# =========================================

print("\nTraining Voting Ensemble...")

voting = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('xgb', xgb),
        ('mlp', mlp)
    ],
    voting='soft'
)

voting.fit(X_train, y_train)

y_pred = voting.predict(X_test)
y_prob = voting.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
auc_score = roc_auc_score(y_test, y_prob, multi_class='ovr')

results.append(["Voting Ensemble", acc, prec, rec, f1, auc_score])

print("\nVoting Ensemble Results:")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)
print("AUC-ROC:", auc_score)

In [ ]:
# =========================================
# 7. RESULTS TABLE
# =========================================

results_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1 Score", "AUC-ROC"
])

print("\nFinal Results:")
print(results_df)

In [ ]:
# =========================================
# 8. CONFUSION MATRICES
# =========================================

for name, model in models.items():
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

ConfusionMatrixDisplay.from_estimator(voting, X_test, y_test)
plt.title("Confusion Matrix - Voting Ensemble")
plt.show()


In [ ]:
# =========================================
# 9. ROC CURVES (MULTI-CLASS)
# =========================================

classes = np.unique(y_test)
y_test_bin = label_binarize(y_test, classes=classes)

for name, model in models.items():
    y_prob = model.predict_proba(X_test)

    plt.figure()
    for i in range(len(classes)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        plt.plot(fpr, tpr, label=f"Class {i}")

    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f"ROC Curve - {name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()

# Ensemble ROC
y_prob = voting.predict_proba(X_test)

plt.figure()
for i in range(len(classes)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, label=f"Class {i}")

plt.plot([0, 1], [0, 1], 'k--')
plt.title("ROC Curve - Voting Ensemble")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()


In [ ]:
# =========================================
# 10. MODEL COMPARISON GRAPH
# =========================================

results_df.plot(
    x="Model",
    y=["Accuracy", "F1 Score", "AUC-ROC"],
    kind="bar",
    figsize=(10, 6)
)

plt.title("Model Comparison")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()